# Stage 2 · Rankings humanos

Analiza las ordenaciones de los codificadores humanos como **referencia
descriptiva** frente a los rankings de los LLM.


## 1. Subir el fichero

In [ ]:
N_PERMUTACIONES = 10000
SEMILLA = 20260301

try:
    from google.colab import files
    print("Sube Appendix_A3_Human_Validation_and_Ranking.xlsx")
    RUTA = list(files.upload().keys())[0]
except ImportError:
    RUTA = "Appendix_A3_Human_Validation_and_Ranking.xlsx"

print("Fichero:", RUTA)

In [ ]:
import pandas as pd
import numpy as np
import re

rng = np.random.default_rng(SEMILLA)
pd.set_option("display.width", 200)

RANGOS = [1, 2, 3, 4, 5, 6]
TAREAS = ["Credibility", "Personal importance", "Societal importance",
          "Willingness to share"]
ORDEN_COND = ["Baseline", "Priming", "Bizarre", "Change", "Confirmation", "Flaws"]
print("Listo")

## 2. Fichero principal



In [ ]:
MAPEO_LETRA = {"Text A": "Baseline", "Text B": "Priming", "Text C": "Bizarre",
               "Text D": "Change", "Text E": "Confirmation", "Text F": "Flaws"}

s2 = pd.read_excel(RUTA, sheet_name="Stage2_Rankings")
s2 = s2.rename(columns={"Coder ID": "coder", "Topic": "topic",
                        "Authority condition": "autoridad", "Task": "tarea"})
cols = [c for c in s2.columns if str(c).startswith("Text ") and "rank" in str(c)]

filas = []
for _, r in s2.iterrows():
    for c in cols:
        filas.append({"coder": r["coder"], "topic": r["topic"],
                      "autoridad": r["autoridad"], "tarea": r["tarea"],
                      "condicion": MAPEO_LETRA[c.replace(" rank", "")],
                      "rango": r[c]})
largo = pd.DataFrame(filas)
print(f"Codificadores en el fichero: {sorted(s2['coder'].dropna().unique())}")
print(f"Filas de ranking: {len(largo)}")

## 3. Que ordenaciones son analizables



In [ ]:
largo["set_id"] = (largo["coder"] + " | " + largo["topic"] + " | "
                   + largo["autoridad"] + " | " + largo["tarea"])

def diagnostico(g):
    v = g["rango"].dropna()
    if len(v) == 0:
        return "vacia"
    if g["condicion"].isna().any():
        return "condicion no identificada"
    if g["condicion"].nunique() != 6:
        return "faltan o sobran condiciones"
    v = sorted(int(x) for x in v)
    if len(v) < 6:
        return "incompleta"
    if v != RANGOS:
        return "rango repetido"
    return "analizable"

estado = (largo.groupby(["set_id", "coder", "topic", "autoridad", "tarea"])
          .apply(diagnostico, include_groups=False)
          .rename("estado").reset_index())

display(pd.crosstab(estado["coder"], estado["estado"], margins=True))

problemas = estado[estado["estado"] == "rango repetido"]
if len(problemas):
    print("\nSets con un rango repetido (excluidos, no recuperables sin preguntar):")
    det = []
    for _, r in problemas.iterrows():
        g = largo[largo["set_id"] == r["set_id"]]
        v = [int(x) for x in g["rango"].dropna()]
        det.append({"coder": r["coder"], "tema": r["topic"],
                    "autoridad": r["autoridad"], "tarea": r["tarea"],
                    "repite": sorted({x for x in v if v.count(x) > 1}),
                    "falta": [x for x in RANGOS if x not in v]})
    display(pd.DataFrame(det))

In [ ]:
analizables = estado.loc[estado["estado"] == "analizable", "set_id"]
val = largo[largo["set_id"].isin(analizables)].copy()
val["rango"] = val["rango"].astype(int)

print(f"Sets analizables : {val['set_id'].nunique()}")
print(f"Codificadores    : {val['coder'].nunique()}  ->  {sorted(val['coder'].unique())}")
print()
print("Sets por codificador:")
print(val.groupby("coder")["set_id"].nunique().to_string())

## 4. Rangos medios por condicion



In [ ]:
tabla = (val.pivot_table(index="tarea", columns="condicion",
                         values="rango", aggfunc="mean")
         .reindex(index=[t for t in TAREAS if t in val["tarea"].unique()],
                  columns=ORDEN_COND).round(2))
display(tabla)

n = val.groupby("tarea")["set_id"].nunique().rename("sets por tarea")
display(n.to_frame())

## 5. Test de permutacion



In [ ]:
def test_permutacion(df, n_perm=N_PERMUTACIONES):
    conds = sorted(df["condicion"].unique())
    esperado = (len(conds) + 1) / 2
    M = np.array([g.set_index("condicion").loc[conds, "rango"].values
                  for _, g in df.groupby("set_id")])
    T_obs = ((M.mean(0) - esperado) ** 2).sum()
    idx = rng.random((n_perm,) + M.shape).argsort(axis=2)
    perm = np.take_along_axis(np.broadcast_to(M, (n_perm,) + M.shape), idx, axis=2)
    T_null = ((perm.mean(1) - esperado) ** 2).sum(1)
    return T_obs, (np.sum(T_null >= T_obs) + 1) / (n_perm + 1), M.shape[0]

def benjamini_hochberg(p):
    p = np.asarray(p, dtype=float)
    o = np.argsort(p)
    r = np.empty_like(o); r[o] = np.arange(1, len(p) + 1)
    adj = p * len(p) / r
    return np.minimum(np.minimum.accumulate(adj[o][::-1])[::-1][np.argsort(o)], 1)

filas = []
for tarea, g in val.groupby("tarea"):
    T, p, n_sets = test_permutacion(g)
    filas.append({"tarea": tarea, "T": round(T, 2), "p": p, "sets": n_sets})
res = pd.DataFrame(filas)
res["p_BH"] = benjamini_hochberg(res["p"]).round(4)
res["p"] = res["p"].round(4)
res["se aparta del azar"] = np.where(res["p_BH"] < .05, "si", "no")
display(res)

## 6. Humanos frente a modelos



In [ ]:
RUTA_GPT = RUTA_LLAMA = None
try:
    from google.colab import files
    print("Sube survey_results_GPT.xlsx (Cancelar para saltar esta seccion)")
    sub = files.upload()
    if sub:
        RUTA_GPT = list(sub.keys())[0]
        print("\nSube survey_results_Ollama.xlsx")
        RUTA_LLAMA = list(files.upload().keys())[0]
except ImportError:
    RUTA_GPT, RUTA_LLAMA = "survey_results_GPT.xlsx", "survey_results_Ollama.xlsx"
except Exception:
    pass
print("GPT:", RUTA_GPT, "| LLaMA:", RUTA_LLAMA)

In [ ]:
PATRONES = [
    ("Change",       r"in comparison to the previous variant|up from last month"),
    ("Priming",      r"how it connects to the old|as repeatedly reported"),
    ("Bizarre",      r"young mother admitted|unusual price changes"),
    ("Confirmation", r"better prepared than you think|everyday purchases may feel"),
    ("Flaws",        r"under-?reacting to the risks|misunderstand what the data"),
    ("Baseline",     r"what we know so far|new data shows"),
]

def condicion_de(texto):
    "El fichero de GPT guarda el titular completo, no la etiqueta."
    s = str(texto).lower()
    for nombre, patron in PATRONES:
        if re.search(patron, s):
            return nombre
    return None

ETIQUETAS = {"BaselineVersion": "Baseline", "BizarreVersion": "Bizarre",
             "ChangedVersion": "Change", "ConfirmationVersion": "Confirmation",
             "FlawsVersion": "Flaws", "PrimedVersion": "Priming"}
TAREA_LLM = {"Credibility": "Credibility", "ImportancePerso": "Personal importance",
             "ImportanceSoc": "Societal importance", "Sharing": "Willingness to share"}

def cond_llm(v):
    if pd.isna(v): return None
    s = str(v).strip()
    if s in ETIQUETAS: return ETIQUETAS[s]
    return condicion_de(s)

def leer_llm(ruta, columnas, modelo):
    d = pd.read_excel(ruta)
    filas = []
    for _, r in d.iterrows():
        sid = f"{modelo}|{r['Order']}|{r['task']}|{r['iteration']}"
        conds = [cond_llm(r[c]) for c in columnas]
        if None in conds or len(set(conds)) != 6:
            continue
        for pos, c in enumerate(conds, start=1):
            filas.append({"modelo": modelo, "set_id": sid,
                          "tarea": TAREA_LLM.get(r["task"], r["task"]),
                          "condicion": c, "rango": pos})
    return pd.DataFrame(filas)

if RUTA_GPT and RUTA_LLAMA:
    llm = pd.concat([leer_llm(RUTA_GPT,   [f"rank_{i}"   for i in range(1,7)], "GPT-4o"),
                     leer_llm(RUTA_LLAMA, [f"rank_{i}_f" for i in range(1,7)], "LLaMA-3.1-8B")])
    perfil_llm = llm.pivot_table(index=["modelo","tarea"], columns="condicion",
                                 values="rango", aggfunc="mean")
    perfil_hum = (val.assign(modelo="Humanos")
                  .pivot_table(index=["modelo","tarea"], columns="condicion",
                               values="rango", aggfunc="mean"))
    comparacion = pd.concat([perfil_hum, perfil_llm])[ORDEN_COND].round(2)
    display(comparacion.sort_index(level="tarea"))

    from scipy.stats import kendalltau
    print("\nAcuerdo entre el perfil humano y el de cada modelo (Kendall tau-b):")
    for tarea in perfil_hum.index.get_level_values("tarea"):
        h = perfil_hum.loc[("Humanos", tarea), ORDEN_COND].values
        for modelo in ["GPT-4o", "LLaMA-3.1-8B"]:
            if (modelo, tarea) in perfil_llm.index:
                m = perfil_llm.loc[(modelo, tarea), ORDEN_COND].values
                tau, p = kendalltau(h, m)
                print(f"  {tarea:22s} vs {modelo:14s} tau = {tau: .2f}  p = {p:.3f}")
else:
    print("Sin ficheros de LLM: seccion omitida.")

## 7. Resumen

In [ ]:
print(f"Sets analizables      : {val['set_id'].nunique()}")
print(f"Codificadores         : {val['coder'].nunique()}")
excl = estado[estado["estado"] != "analizable"]["estado"].value_counts()
print(f"Sets excluidos        : {len(estado) - int((estado['estado']=='analizable').sum())}")
for motivo, n_ in excl.items():
    print(f"    {motivo:28s} {n_}")
print()
print("Dimensiones que se apartan de la intercambiabilidad:")
for _, r in res.iterrows():
    marca = "SI" if r["p_BH"] < .05 else "no"
    print(f"  {r['tarea']:22s} T = {r['T']:5.2f}  p_BH = {r['p_BH']:.4f}   {marca}")